# Text preprocessing & normalization — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Normalization as a controlled equivalence relation over text
These five cells show how lowercasing, Unicode normalization, punctuation policy, stop-word policy and sequence padding change a toy corpus while keeping the mapping auditable.

In [ ]:
text="Café CAFÉ cafe!"
raw=list(text); norm=text.casefold().replace("é","e").replace("!","")
counts=[len(text), len(norm), len(set(raw)), len(set(norm))]
plt.figure(figsize=(6,3)); plt.bar(['raw chars','norm chars','raw vocab','norm vocab'],counts); plt.title('Normalization reduces accidental distinctions')
assert norm=='cafe cafe cafe'

In [ ]:
import unicodedata
forms=["Café", "Café"]
nfc=[unicodedata.normalize('NFC', s) for s in forms]
lengths=[len(s) for s in forms+nfc]
plt.figure(figsize=(6,3)); plt.bar(['decomp','precomp','NFC1','NFC2'],lengths); plt.title('Unicode NFC makes equivalent spellings identical')
assert nfc[0]==nfc[1] and lengths[:2]==[5,4]

In [ ]:
sent="Win $5,000 now!!!"
keep_digits=''.join(ch.casefold() for ch in sent if ch.isalnum() or ch.isspace())
drop_digits=''.join(ch.casefold() for ch in sent if ch.isalpha() or ch.isspace())
plt.figure(figsize=(6,3)); plt.bar(['keep digits','drop digits'],[len(keep_digits.split()),len(drop_digits.split())]); plt.title('Punctuation policy can preserve or erase signal')
assert keep_digits.split()==['win','5000','now'] and drop_digits.split()==['win','now']

In [ ]:
tokens="the model is not bad".split(); stop={'the','is'}
kept=[t for t in tokens if t not in stop]
wrong=[t for t in tokens if t not in stop|{'not'}]
plt.figure(figsize=(6,3)); plt.bar(['safe stoplist','drops not'],[len(kept),len(wrong)]); plt.title('Stop-word policy changes meaning-bearing length')
assert kept==['model','not','bad'] and wrong==['model','bad']

In [ ]:
seqs=["short text".split(), "a much longer text".split(), "tiny".split()]
maxlen=4; padded=[s+['<pad>']*(maxlen-len(s)) for s in seqs]
mask=np.array([[tok!='<pad>' for tok in row] for row in padded])
plt.figure(figsize=(6,3)); plt.imshow(mask, aspect='auto', cmap='Greens'); plt.yticks(range(3),['s1','s2','s3']); plt.title('Padding creates a mask, not real tokens')
assert mask.sum()==7 and padded[0][-1]=='<pad>'